# Module 6 - Alzheimer's: Statistical Analysis (New Techniques)

This is the workhorse notebook. Every technique here is one we have NOT used in the 5-FU module. Each section is self-contained.

**Techniques covered:**
1. **Bayesian Information Component (IC / BCPNN)** - the WHO Uppsala Monitoring Centre's Bayesian alternative to ROR. Handles small cell counts via shrinkage; the IC and its 95% credible interval are the standard signal-detection metrics for the WHO VigiBase.
2. **Cochran-Armitage trend test** - tests whether the proportion of serious outcomes trends monotonically across an *ordered* categorical variable (age band). More powerful than plain chi-square when the ordering matters.
3. **Multivariable logistic regression** - adjusted odds ratios for `serious outcome ~ drug_class + age + sex`, so drug-class effects can be reported after controlling for age and sex confounders.
4. **Kruskal-Wallis + pairwise Mann-Whitney with Bonferroni** - nonparametric ANOVA analog. Tests whether the *distribution* of reactions-per-report differs across drug classes without assuming normality.
5. **Bootstrap 95% CI for ROR** - nonparametric confidence interval that doesn't rely on the log-normal approximation. Good sanity check on the log-normal ROR CIs we've been using elsewhere in the project.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

db_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\FDA_FAERS\database\faers.db"
conn = sqlite3.connect(db_path)
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

## 1. Bayesian Information Component (IC / BCPNN)

The **Information Component** is a Bayesian log-ratio between the observed rate of a drug-reaction pair and the rate expected if they were independent.

Formula (with Bayesian shrinkage priors that stabilize small counts):

    IC = log2( (a + 0.5) * (N + 2) / ((a + b + 1) * (a + c + 1)) )

    Var(IC) approx = 1/ln(2)^2 * ( 1/(a+0.5) + 1/(a+b+1) + 1/(a+c+1) - 3/(N+2) )

    IC025 = IC - 1.96 * sqrt(Var(IC))
    IC975 = IC + 1.96 * sqrt(Var(IC))

Where:
- `a` = reports with both the drug AND the reaction
- `b` = reports with the drug but NOT the reaction
- `c` = reports with the reaction but NOT the drug
- `N` = total reports in the database

**Rule of thumb:** `IC025 > 0` is the WHO signal threshold (the 2.5th percentile of the posterior is still above the independence line).

Computed here for the top 25 reactions in the whole Alzheimer's cohort vs the rest of FAERS.

In [ ]:
# Cell counts: a, b, c across all reactions of interest.
alz_reports_total  = pd.read_sql_query(
    'SELECT COUNT(DISTINCT primaryid) AS n FROM alzheimers_analysis', conn
).iloc[0, 0]
faers_reports_total = pd.read_sql_query(
    'SELECT COUNT(DISTINCT primaryid) AS n FROM demo', conn
).iloc[0, 0]

a_counts = pd.read_sql_query("""
    SELECT r.pt AS reaction, COUNT(DISTINCT r.primaryid) AS a
    FROM reac r
    JOIN (SELECT DISTINCT primaryid FROM alzheimers_analysis) alz
        ON r.primaryid = alz.primaryid
    GROUP BY r.pt
    ORDER BY a DESC
    LIMIT 25
""", conn)

placeholders = ','.join(['?'] * len(a_counts))
c_counts = pd.read_sql_query(f"""
    SELECT r.pt AS reaction, COUNT(DISTINCT r.primaryid) AS with_reaction_total
    FROM reac r
    WHERE r.pt IN ({placeholders})
    GROUP BY r.pt
""", conn, params=a_counts['reaction'].tolist())

ic_table = a_counts.merge(c_counts, on='reaction')
ic_table['b'] = alz_reports_total - ic_table['a']
ic_table['c'] = ic_table['with_reaction_total'] - ic_table['a']
N = faers_reports_total

# BCPNN Information Component + 95% credible interval
ic_table['IC'] = np.log2(
    (ic_table['a'] + 0.5) * (N + 2)
    / ((ic_table['a'] + ic_table['b'] + 1) * (ic_table['a'] + ic_table['c'] + 1))
)
ic_table['var_IC'] = (1.0 / np.log(2) ** 2) * (
    1 / (ic_table['a'] + 0.5)
    + 1 / (ic_table['a'] + ic_table['b'] + 1)
    + 1 / (ic_table['a'] + ic_table['c'] + 1)
    - 3 / (N + 2)
)
ic_table['IC025'] = ic_table['IC'] - 1.96 * np.sqrt(ic_table['var_IC'])
ic_table['IC975'] = ic_table['IC'] + 1.96 * np.sqrt(ic_table['var_IC'])
ic_table['signal'] = ic_table['IC025'] > 0

ic_signals = (ic_table[ic_table['signal']]
              .sort_values('IC', ascending=False)
              [['reaction', 'a', 'IC', 'IC025', 'IC975']]
              .round(2))
print(f'{len(ic_signals)} of top-25 reactions cross the IC025 > 0 signal threshold\n')
ic_signals

In [ ]:
# Forest plot of IC + credible interval, sorted by IC.
plot = ic_table.sort_values('IC')
fig, ax = plt.subplots(figsize=(9, 8))
ax.errorbar(plot['IC'], range(len(plot)),
            xerr=[plot['IC'] - plot['IC025'], plot['IC975'] - plot['IC']],
            fmt='o', color='steelblue', capsize=4)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1)
ax.set_yticks(range(len(plot)))
ax.set_yticklabels(plot['reaction'])
ax.set_xlabel('Information Component (IC)  [IC025 > 0 = signal]')
ax.set_title('BCPNN disproportionality - Alzheimer\'s cohort vs rest of FAERS')
plt.tight_layout()
plt.show()

## 2. Cochran-Armitage trend test

Chi-square only tells you whether two categorical variables are related. When one variable is *ordered* (like age band), the Cochran-Armitage trend test is more powerful because it looks specifically for a monotone trend.

**Question:** Does the proportion of serious-outcome reports rise (or fall) monotonically with age band, within the Alzheimer's cohort?

`scipy.stats` doesn't have Cochran-Armitage directly, so it's implemented from scratch. The test statistic is asymptotically standard normal.

In [ ]:
def cochran_armitage(counts_pos, counts_total, scores=None):
    """
    counts_pos:   1D array of positive counts per ordered group.
    counts_total: 1D array of total counts per ordered group.
    scores:       weights per group; defaults to 0, 1, 2, ...

    Returns z-statistic and 2-sided p-value.
    """
    counts_pos   = np.asarray(counts_pos,   dtype=float)
    counts_total = np.asarray(counts_total, dtype=float)
    if scores is None:
        scores = np.arange(len(counts_pos), dtype=float)
    else:
        scores = np.asarray(scores, dtype=float)

    N       = counts_total.sum()
    total_p = counts_pos.sum()
    p_bar   = total_p / N

    num = np.sum(scores * (counts_pos - counts_total * p_bar))
    var = p_bar * (1 - p_bar) * (
        np.sum(counts_total * scores ** 2)
        - (np.sum(counts_total * scores) ** 2) / N
    )
    z = num / np.sqrt(var)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p


ca_data = pd.read_sql_query("""
    SELECT d.age,
           MAX(CASE WHEN o.outc_cod IN ('DE','LT','HO','DS','CA') THEN 1 ELSE 0 END) AS serious
    FROM demo d
    JOIN (SELECT DISTINCT primaryid FROM alzheimers_analysis) a
        ON d.primaryid = a.primaryid
    LEFT JOIN outc o ON o.primaryid = a.primaryid
    WHERE d.age BETWEEN 40 AND 110
      AND d.age_cod = 'YR'
    GROUP BY d.primaryid
""", conn)

ca_data['age_group'] = pd.cut(
    ca_data['age'],
    bins=[40, 55, 65, 75, 85, 110],
    labels=['40-54', '55-64', '65-74', '75-84', '85+']
)

trend_tbl = (ca_data
             .groupby('age_group', observed=True)
             .agg(total=('serious', 'size'), positive=('serious', 'sum'))
             .assign(pct_serious=lambda x: (x['positive'] / x['total'] * 100).round(1)))
print(trend_tbl)

z, p = cochran_armitage(trend_tbl['positive'].values,
                        trend_tbl['total'].values)
print(f'\nCochran-Armitage z = {z:.3f},  p = {p:.4g}')
if p < 0.05:
    print('Monotone trend across age bands is statistically significant.')
else:
    print('No significant monotone trend detected.')

## 3. Multivariable logistic regression

Serious-outcome comparisons across drug classes are confounded by age and sex (older patients + certain sex ratios per class). Multivariable logistic regression gives an odds ratio for each drug class *after adjusting* for age and sex.

Model:  `logit(P(serious=1)) = beta_0 + beta_class * drug_class + beta_age * age + beta_sex * sex`

Interpretation: `exp(beta_class)` is the adjusted odds ratio for that class vs the reference class (ChEI here, alphabetically first).

In [ ]:
model_df = pd.read_sql_query("""
    SELECT a.primaryid,
           MAX(a.drug_class)   AS drug_class,
           MAX(d.age)          AS age,
           MAX(d.sex)          AS sex,
           MAX(CASE WHEN o.outc_cod IN ('DE','LT','HO','DS','CA')
                    THEN 1 ELSE 0 END) AS serious
    FROM alzheimers_analysis a
    JOIN demo d ON d.primaryid = a.primaryid
    LEFT JOIN outc o ON o.primaryid = a.primaryid
    WHERE d.age BETWEEN 40 AND 110
      AND d.age_cod = 'YR'
      AND d.sex IN ('M','F')
    GROUP BY a.primaryid
""", conn)

# Drop the tiny anti-amyloid group only if it truly breaks the fit; otherwise keep it.
print(model_df['drug_class'].value_counts())

logit_fit = smf.logit(
    'serious ~ C(drug_class, Treatment(reference="ChEI")) + age + C(sex)',
    data=model_df
).fit(disp=0)

print(logit_fit.summary())

# Adjusted odds ratios with 95% CIs
params = logit_fit.params
conf   = logit_fit.conf_int()
or_tbl = pd.DataFrame({
    'aOR':      np.exp(params),
    'CI_lower': np.exp(conf[0]),
    'CI_upper': np.exp(conf[1]),
    'p':        logit_fit.pvalues,
})
or_tbl.round(3)

## 4. Kruskal-Wallis + pairwise Mann-Whitney with Bonferroni

**Question:** Does the *number of reactions per report* differ across drug classes?

Reaction counts are right-skewed and bounded below at 1, so a parametric ANOVA is inappropriate. Kruskal-Wallis is the nonparametric equivalent; if it rejects, do pairwise Mann-Whitney with Bonferroni correction to see which pairs of classes differ.

In [ ]:
from itertools import combinations

rx_per_report = pd.read_sql_query("""
    SELECT a.primaryid,
           MAX(a.drug_class) AS drug_class,
           COUNT(DISTINCT r.pt) AS n_reactions
    FROM alzheimers_analysis a
    JOIN reac r ON r.primaryid = a.primaryid
    GROUP BY a.primaryid
""", conn)

summary = (rx_per_report
           .groupby('drug_class')['n_reactions']
           .describe()[['count','mean','50%','75%','max']]
           .rename(columns={'50%': 'median', '75%': 'p75'}))
print(summary.round(2))

groups = [g['n_reactions'].values for _, g in rx_per_report.groupby('drug_class')]
group_names = list(rx_per_report.groupby('drug_class').groups.keys())
h, p = stats.kruskal(*groups)
print(f'\nKruskal-Wallis H = {h:.2f},  p = {p:.4g}')

# Pairwise Mann-Whitney with Bonferroni
pairs = list(combinations(range(len(groups)), 2))
n_tests = len(pairs)
print(f'\nPairwise Mann-Whitney (Bonferroni-adjusted alpha = 0.05 / {n_tests} = {0.05/n_tests:.4f})')
for i, j in pairs:
    u, pv = stats.mannwhitneyu(groups[i], groups[j], alternative='two-sided')
    adj = min(pv * n_tests, 1.0)
    flag = '  *' if adj < 0.05 else ''
    print(f'  {group_names[i]:<15s} vs {group_names[j]:<15s}  U = {u:>10.0f}  raw p = {pv:.4g}  adj p = {adj:.4g}{flag}')

## 5. Bootstrap 95% CI for a Reporting Odds Ratio

The log-normal ROR CI (used in modules 1 and 4) assumes large-sample normality of log(ROR). For small cell counts, that assumption breaks. A nonparametric bootstrap resamples the data with replacement, recomputes ROR each time, and takes the 2.5 / 97.5 percentiles of the resulting distribution.

Example: bootstrap ROR for **Fall** in the Alzheimer's cohort vs the rest of FAERS. Falls are a classic AD-drug adverse event.

In [ ]:
TARGET_REACTION = 'Fall'
N_BOOT = 2000

# Build a report-level dataset: is_alz (drug flag) and has_reaction.
data = pd.read_sql_query("""
    WITH alz AS (SELECT DISTINCT primaryid FROM alzheimers_analysis)
    SELECT d.primaryid,
           CASE WHEN alz.primaryid IS NOT NULL THEN 1 ELSE 0 END AS is_alz,
           MAX(CASE WHEN r.pt = ? THEN 1 ELSE 0 END)             AS has_reaction
    FROM demo d
    LEFT JOIN alz  ON alz.primaryid = d.primaryid
    LEFT JOIN reac r ON r.primaryid  = d.primaryid
    GROUP BY d.primaryid
""", conn, params=(TARGET_REACTION,))

def compute_ror(df):
    a = ((df['is_alz'] == 1) & (df['has_reaction'] == 1)).sum()
    b = ((df['is_alz'] == 1) & (df['has_reaction'] == 0)).sum()
    c = ((df['is_alz'] == 0) & (df['has_reaction'] == 1)).sum()
    d = ((df['is_alz'] == 0) & (df['has_reaction'] == 0)).sum()
    if b == 0 or c == 0:
        return np.nan
    return (a * d) / (b * c)

point_est = compute_ror(data)
print(f'Point ROR for {TARGET_REACTION}: {point_est:.3f}')

rng = np.random.default_rng(seed=42)
boot_rors = []
n = len(data)
for _ in range(N_BOOT):
    idx = rng.integers(0, n, size=n)
    boot_rors.append(compute_ror(data.iloc[idx]))

boot_rors = np.array([r for r in boot_rors if not np.isnan(r)])
lo, hi = np.percentile(boot_rors, [2.5, 97.5])
print(f'Bootstrap 95% CI ({N_BOOT} resamples): [{lo:.3f}, {hi:.3f}]')

plt.figure(figsize=(8, 4))
plt.hist(boot_rors, bins=40, color='steelblue', edgecolor='white')
plt.axvline(point_est, color='red', linestyle='--', label=f'Point ROR = {point_est:.2f}')
plt.axvline(lo, color='black', linestyle=':', label=f'2.5% = {lo:.2f}')
plt.axvline(hi, color='black', linestyle=':', label=f'97.5% = {hi:.2f}')
plt.axvline(1.0, color='gray', linestyle='-', label='Null (ROR = 1)')
plt.xlabel('Bootstrap ROR')
plt.ylabel('Frequency')
plt.title(f'Bootstrap distribution: ROR for {TARGET_REACTION} in Alzheimer\'s cohort')
plt.legend()
plt.tight_layout()
plt.show()